# RAG(Retriever Agumented Generation)==
1. Retrieval
When a user asks a question, the system does not immediately send it to the LLM to guess the answer. Instead, it converts the question into a format the system understands (usually a vector) and searches an external database (like a Vector Database) to retrieve the most relevant documents or chunks of text related to the query.

2. Augmented
The system then takes the user's original question and augments (enhances) it by attaching the relevant information it just retrieved. It creates a new, rich prompt that essentially says: "Here is the user's question, and here is some factual context I found in our database. Use this context to answer the question."

3. Generation
Finally, this augmented prompt is sent to the LLM. The model reads the provided context and generates a final answer based only on those specific facts, rather than pulling from its broad, general training data.

RAG pipeline architecture

User Query
     ↓
Query Processing
     ↓
Retriever (Vector DB / Hybrid Search)
     ↓
Relevant Documents
     ↓
Prompt Construction (Context + Question)
     ↓
LLM Generator
     ↓
Final Answer

## Retrieval Failures
Retrieval Failure Modes Taxonomy

Understanding failures is critical.

1. Recall Failures (Nothing Relevant Retrieved)

Cause:

Poor embeddings
Bad chunking
Vocabulary mismatch
Low k value

Example:
Query: “How do we handle memory leaks?”
Docs say: “Garbage collection tuning strategies”



2. Precision Failures (Wrong Docs Retrieved)

Cause:

Overlapping embeddings
Generic chunks
No metadata filtering

Example:
Query: “Apple revenue 2023”
Retrieved: Apple fruit farming article.

3. Chunking Failures

Context split across chunks
Important information cut mid-sentence
Headers separated from body

4. Embedding Model Mismatch

Using general embeddings for domain-specific corpus
Code embeddings for legal text

5. Hallucination Despite Retrieval

Weak grounding prompt
LLM ignores context
Conflicting chunks

6. Metadata Filtering Errors

Incorrect tagging during ingestion
Filter too strict → zero results
Filter too loose → noise


## LOAD the Documents wither it is pdf or txt file 

In [1]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, PyPDFLoader

documents = []
data_dir = Path("./data")

for file in data_dir.iterdir():

    if file.suffix.lower() == ".txt":
        print(f"Loading TXT: {file}")
        documents.extend(TextLoader(str(file)).load())

    elif file.suffix.lower() == ".pdf":
        print(f"Loading PDF: {file}")
        documents.extend(PyPDFLoader(str(file)).load())

print(f"Total documents loaded: {len(documents)}")


/tmp/ipykernel_2982/638227778.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader
/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading PDF: data/Retrieval-Augmented_Generation_RAG.pdf
Total documents loaded: 12


## Chunking the documents 

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=splitter.split_documents(documents)
print("Splitter created Successfully")
print(f"length of chunks:{len(chunks)}")


Splitter created Successfully
length of chunks:152


In [12]:
for chunk in chunks:
    chunk.metadata["topic"] = "Retrieval"
    chunk.metadata["document_type"] = "research_paper"
print(chunks[0].metadata)

{'producer': 'iText® 5.5.13.3 ©2000-2022 iText Group NV (SPRINGER SBM; licensed version)', 'creator': 'PyPDF', 'creationdate': '2025-09-18T03:03:54+02:00', 'moddate': '2025-09-18T03:03:54+02:00', 'source': 'data/Retrieval-Augmented_Generation_RAG.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'topic': 'Retrieval', 'document_type': 'research_paper'}


## Embedding + store 

In [13]:
from sentence_transformers import SentenceTransformer

model= SentenceTransformer("all-MiniLM-L6-v2")
texts=[chunk.page_content for chunk in chunks]

embeddings=model.encode(
            texts,
            convert_to_numpy=True
        )
print(embeddings.shape)
# collection.add(
#     ids=[f"chunk_{i}" for i in range(len(chunks))],
#     documents=texts,
#     embeddings=embeddings.tolist(),
#     metadatas=[chunk.metadata for chunk in chunks]
# )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 461.77it/s]


(152, 384)


In [14]:
import chromadb
client = chromadb.PersistentClient(path='./chroma_db')

collection= client.get_or_create_collection( name= "documents")

collection.add(
     ids=[f"doc_{i}" for i in range(len(texts))],
    documents=texts,
    embeddings=embeddings.tolist(),
    # ids=[f"doc_{i}" for i in range(len(texts))]
    metadatas=[chunk.metadata for chunk in chunks]
)

print("Documents loaded successfully!")

Documents loaded successfully!


In [5]:
# from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_chroma import Chroma

# embedding_model = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

# vectorstore = Chroma.from_documents(
#     documents=chunks,
#     embedding=embedding_model,
#     persist_directory="./chroma_db"
# )

# print("Vector store created successfully!")

## retreive the data 

In [15]:
query = "What are the challenges of RAG?"
# What are the components of a RAG architecture?
# What is the fundamental architecture of RAG?
# what is RAG architecture pipeline?

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=8
)

for i, doc in enumerate(results["documents"][0], start=1):
    print(f"\nchunk {i}")
    print(doc)
    print("="*50)



chunk 1
riegel et al. 2024). We contribute to this ongoing engage-
ment with current AI developments by focusing on RAG.
Speciﬁcally, we review the fundamental architecture of
RAG and highlight some extensions that can enhance a
plain vanilla RAG architecture. We showcase how RAG
can be used in different use-case scenarios and summarize
the most important advantages and challenges that should
be considered when using RAG and RAG-speciﬁc exten-
sions. Finally, we discuss important research avenues for

chunk 2
RAG is an effective way to add this knowledge. By adding
additional information, questions can be answered using
this data, reducing the likelihood of inaccurate answers.
Thus, RAG is an effective measure to enhance factual
accuracy.
A RAG architecture allows references to be provided to
the contextual data stored in the vector database. Providing
valid references to the generated result has been termed
grounding (Magesh et al. 2024). Grounding is a signiﬁcant

chunk 3
overview o

In [16]:
print((results["documents"][0][4]))

issue.
Retrieval effectiveness The effectiveness of a RAG architecture depends on how effectively the retrieval mechanism works.
This includes the effectiveness of the document ranking (i.e., are the most relevant documents ranked
ﬁrst?) and how well the retrieval process performs.
Fig. 3 Research questions related to RAG
123
558 M. Klesel, H. F. Wittmann: Retrieval-Augmented Generation (RAG), Bus Inf Syst Eng 67(4):551–561 (2025)


In [17]:
for i, doc in enumerate(results["documents"][0], start=1):
    print(f"Chunk {i}: {len(doc)} characters")

Chunk 1: 497 characters
Chunk 2: 464 characters
Chunk 3: 478 characters
Chunk 4: 460 characters
Chunk 5: 434 characters
Chunk 6: 453 characters
Chunk 7: 484 characters
Chunk 8: 472 characters


In [9]:
# keywords = ["challenge"]

# for chunk in texts:
#     text = chunk.lower()

#     if all(word in text for word in keywords):
#         print(chunk[:500])
#         print("=" * 50)

## Generate or load the LLM model!

In [21]:
import boto3
import json
import os
from dotenv import load_dotenv

# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv( "AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

load_dotenv("myenv.env")

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)
MODEL_ID = "amazon.nova-micro-v1:0"

In [22]:
prompt = f"""

Context:
{chr(10).join(results["documents"][0])}

Question:
{query}


Answer based on the context. If partially available,
summarize whatever relevant information you can find.
"""

body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 512,
        "temperature": 0.2
    }
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json",
)
response = json.loads(response["body"].read())


print(response["output"]["message"]["content"][0]["text"])



Based on the provided context, the challenges of Retrieval-Augmented Generation (RAG) can be summarized as follows:

1. **Comprehensive Understanding Limitations**:
   - The use of RAG with rich data still has limitations in terms of comprehensive understanding, particularly when dealing with contextually unique terms or complex fictional universes. This suggests that even with RAG, there might be gaps in fully grasping the context and nuances of certain documents.

2. **Retrieval Effectiveness**:
   - The effectiveness of a RAG architecture depends heavily on the efficiency and accuracy of the retrieval mechanism. This includes the effectiveness of document ranking and the overall performance of the retrieval process. If the retrieval mechanism is not optimal, it can lead to less relevant documents being retrieved, which can affect the quality of the generated responses.

3. **Contextual Dependency**:
   - RAG's performance is highly dependent on the context in which it is applied. Fo

## without Filter 
Based on the provided context, the challenges of Retrieval-Augmented Generation (RAG) can be summarized as follows:

1. **Comprehensive Understanding Limitations**:
   - The use of RAG with rich data still has limitations in terms of comprehensive understanding, especially when dealing with highly contextualized documents or complex fictional universes. This is because RAG may struggle to fully grasp the nuances and intricacies of such specialized contexts.

2. **Retrieval Effectiveness**:
   - The effectiveness of a RAG architecture is contingent upon the efficiency and accuracy of its retrieval mechanism. This includes the effectiveness of document ranking and the overall performance of the retrieval process. If the retrieval mechanism does not effectively identify and prioritize the most relevant documents, the system's performance can be compromised.

3. **Contextual Dependency**:
   - RAG architectures can be heavily dependent on the context in which they are used. This means that their performance may vary significantly across different domains and applications, particularly when the context is unique or highly specialized.


## with Filter (metadata filter = RAG)
Based on the provided context, the challenges of Retrieval-Augmented Generation (RAG) can be summarized as follows:

1. **Comprehensive Understanding Limitations**:
   - Even with rich data, RAG-based architectures still face limitations in terms of comprehensive understanding, especially when dealing with context-specific terms unique to particular domains or narratives (e.g., fictional universes with magic and fictional characters).

2. **Retrieval Effectiveness**:
   - The effectiveness of a RAG architecture is heavily dependent on the efficiency and accuracy of the retrieval mechanism. This includes the effectiveness of document ranking and the overall performance of the retrieval process.

3. **Contextual Specificity**:
   - In scenarios requiring context-specific answers, there may be challenges in ensuring that the retrieved information is both relevant and sufficiently detailed to provide accurate responses.

4. **Real-Time Performance**:
   - Ensuring real-time performance can be a challenge, particularly in applications where the retrieval and generation processes need to be fast and responsive.

5. **Incorporation of External Data**:
   - While RAG allows for the incorporation of external and contemporary materials, managing and integrating this data effectively without overwhelming the system or introducing noise can be challenging.

## metadata filter(filter = Retrieval)

Based on the provided context, the challenges of Retrieval-Augmented Generation (RAG) can be summarized as follows:

1. **Comprehensive Understanding Limitations**:
   - The use of RAG with rich data still has limitations in terms of comprehensive understanding, particularly when dealing with contextually unique terms or complex fictional universes. This suggests that even with RAG, there might be gaps in fully grasping the context and nuances of certain documents.

2. **Retrieval Effectiveness**:
   - The effectiveness of a RAG architecture depends heavily on the efficiency and accuracy of the retrieval mechanism. This includes the effectiveness of document ranking and the overall performance of the retrieval process. If the retrieval mechanism is not optimal, it can lead to less relevant documents being retrieved, which can affect the quality of the generated responses.

3. **Contextual Dependency**:
   - RAG's performance is highly dependent on the context in which it is applied. For example, in applications requiring context-specific answers, the effectiveness of RAG can vary based on the richness and relevance of the contextual data available in the vector database.

4. **Real-Time Performance**:
   - For real-time applications, ensuring that RAG can quickly and accurately retrieve and integrate relevant information is crucial. Delays in retrieval or processing can hinder the system's effectiveness.

5. **Integration of External Data**:
   - While RAG allows for the incorporation of external and contemporary materials, the system must effectively manage and integrate this new data without disrupting the existing knowledge base. This can be challenging, especially when the new data is significantly different from the pre-existing context